# Clinical Correlation Analysis

This notebook implements correlation analyses between brain measurements and clinical questionnaire scores in IBS patients.

## Overview

The analysis pipeline includes:

1. **Clinical Data Processing**
   - Loads questionnaire data from UK Biobank
   - Calculates clinical scores:
     - IBS-SSS (IBS Symptom Severity Score)
     - PHQ-12 (Patient Health Questionnaire)
     - GAD-7 (Generalized Anxiety Disorder)
     - PHQ-9 (Patient Health Questionnaire)

2. **Data Integration**
   - Merges clinical scores with brain measurements
   - Handles missing values
   - Creates analysis-ready datasets

3. **Correlation Analysis**
   - Computes correlations between:
     - Brain measurements
     - Clinical scores
     - Demographic variables
   - Applies appropriate statistical tests
   - Corrects for multiple comparisons

4. **Visualization**
   - Generates correlation plots
   - Creates summary tables
   - Visualizes significant findings

## Usage Instructions

1. Set the correct paths in the first code cell:
   - `root_dir`: Path to project root
   - `raw_data_dir`: Path to raw UKB data
   - `docu_dir`: Path to documentation files
   - `data_dir`: Path to preprocessed data
   - `model_dir`: Path to normative model outputs
   - `out_dir`: Path for analysis results

2. Run cells sequentially to:
   - Load and process clinical data
   - Calculate questionnaire scores
   - Perform correlation analyses
   - Generate visualizations

## Key Parameters

- `perm`: Number of permutations for cross-validation
- `clinical_item`: List of questionnaire items to include
- `response_to_score`: Mapping of responses to scores

## Output Files

The analysis generates:
- Clinical score calculations
- Correlation results
- Visualization plots
- Summary statistics

In [1]:
import os
import pandas as pd

perm = 1
root_dir = 'Abosolute path to this project'
raw_data_dir = 'Absolute data path'
rawdata_csv = os.path.join(raw_data_dir, 'ukb52200.csv')
docu_dir = os.path.join(root_dir, '1_document')
data_dir = os.path.join(root_dir, '3_rerun_whole_work', '1_data_cleaned')
model_dir = os.path.join(root_dir, '3_rerun_whole_work', '2_models_sMRI')
out_dir = os.path.join(root_dir, '3_rerun_whole_work', '5_correlation')
os.makedirs(out_dir,exist_ok=True)

# Clinical questionnaire

## Questionnaire items inclusion

In [ ]:
HC_reference = pd.read_csv(os.path.join(data_dir, 'HC_reference_MRI_age.csv'), usecols=['eid', '31-0.0', '21003-2.0'])
HC_cov = pd.read_csv(os.path.join(data_dir, 'HC_MRI_age.csv'), usecols=['eid', '31-0.0', '21003-2.0'])
IBS_cov = pd.read_csv(os.path.join(data_dir, 'IBS_All_MRI_age.csv'), usecols=['eid', '31-0.0', '21003-2.0', 'Group'])
HC_reference.rename(columns={'21003-2.0':'age', '31-0.0':'sex'}, inplace=True)
HC_cov.rename(columns={'21003-2.0':'age', '31-0.0':'sex'}, inplace=True)
IBS_cov.rename(columns={'21003-2.0':'age', '31-0.0':'sex'}, inplace=True)

clinical_item = ['eid'] + ['210'+str(icd_array)+'-0.0' for icd_array in range(24, 62)] + ['20506-0.0', '20509-0.0', '20520-0.0', '20515-0.0', 
                                                                                          '20516-0.0', '20505-0.0', '20512-0.0', '20514-0.0', 
                                                                                          '20510-0.0', '20517-0.0', '20519-0.0', '20511-0.0', 
                                                                                          '20507-0.0', '20508-0.0', '20518-0.0', '20513-0.0', '21001-2.0']
clinical_data = pd.read_csv(rawdata_csv, chunksize=100000, usecols=clinical_item)
HC_reference_clinical = pd.DataFrame()
HC_clinical = pd.DataFrame()
IBS_clinical = pd.DataFrame()
for chunk in clinical_data:
    HC_reference_clinical = pd.concat([HC_reference_clinical, pd.merge(HC_reference, chunk, how='inner', on='eid')], axis=0)
    HC_clinical = pd.concat([HC_clinical, pd.merge(HC_cov, chunk, how='inner', on='eid')], axis=0)
    IBS_clinical = pd.concat([IBS_clinical, pd.merge(IBS_cov, chunk, how='inner', on='eid')], axis=0)
HC_reference_clinical.rename(columns={'21001-2.0':'BMI'}, inplace=True)
HC_clinical.rename(columns={'21001-2.0':'BMI'}, inplace=True)
IBS_clinical.rename(columns={'21001-2.0':'BMI'}, inplace=True)
HC_reference_clinical.to_csv(os.path.join(data_dir, 'HC_reference_clinical.csv'),index=False)
HC_clinical.to_csv(os.path.join(data_dir, 'HC_clinical.csv'),index=False)
IBS_clinical.to_csv(os.path.join(data_dir, 'IBS_clinical.csv'),index=False)

## Questionnaire score calculation

### IBS-SSS

In [3]:
HC_reference_clinical = pd.read_csv(os.path.join(data_dir, 'HC_reference_clinical.csv'))
HC_clinical = pd.read_csv(os.path.join(data_dir, 'HC_clinical.csv'))
IBS_clinical = pd.read_csv(os.path.join(data_dir, 'IBS_clinical.csv'))

# Define the columns based on the UK Biobank coding
prompt_columns = ['21035-0.0', '21035-0.0', '21038-0.0']  # Prompt columns for subscores 1-3
score_columns = ['21036-0.0', '21037-0.0', '21039-0.0']   # Score columns for subscores 1-3
other_columns = ['21040-0.0', '21041-0.0']                # Subscores 4 and 5

# Function to calculate IBS-SSS score
def calculate_ibsss_score(df):
    # Check if all required columns are present
    if all(col in df.columns for col in prompt_columns + other_columns):
        # Initialize the IBS-SSS score with None for rows missing any required columns
        df['IBS-SSS'] = df.apply(lambda row: None if any(pd.isna(row[col]) for col in prompt_columns + other_columns) else 0, axis=1)
        
        # Calculate scores for subscores 1-3
        for prompt_col, score_col in zip(prompt_columns, score_columns):
            df['IBS-SSS'] += df.apply(lambda row: row[score_col] if row[prompt_col] == 1 and row[score_col] >= 0 else 0, axis=1)
        
        # Add scores for subscores 4 and 5
        df['IBS-SSS'] += df[other_columns].clip(lower=0).sum(axis=1)
    else:
        print("Some columns are missing in the dataframe.")
    return df

# Apply the function to each dataframe
HC_reference_clinical = calculate_ibsss_score(HC_reference_clinical)
HC_clinical = calculate_ibsss_score(HC_clinical)
IBS_clinical = calculate_ibsss_score(IBS_clinical)

### PHQ-12

In [4]:
# Define the columns based on the UK Biobank coding
phq12_columns = [
    '21048-0.0',  # Back pain
    '21051-0.0',  # Headaches
    '21052-0.0',  # Chest pain
    '21053-0.0',  # Dizziness
    '21054-0.0',  # Fainting spells
    '21055-0.0',  # Heart pounding
    '21056-0.0',  # Shortness of breath
    '21057-0.0',  # Pain during intercourse
    '21049-0.0',  # Limb pain
    '21060-0.0',  # Feeling tired
    # '21050-0.0',  # Menstrual cramps
    '21061-0.0'   # Trouble sleeping
]

# Mapping of responses to scores
response_to_score = {
    -600: 0,  # Not bothered at all
    -601: 1,  # Bothered a little
    -602: 2   # Bothered a lot
}

# Function to calculate PHQ-12 score
def calculate_phq12_score(df):
    # Check if all required columns are present
    if all(col in df.columns for col in phq12_columns[:-2] + [phq12_columns[-1]]):
        # Map responses to scores and sum them
        df['PHQ-12'] = df[phq12_columns].applymap(lambda x: 0 if x < -800 else response_to_score.get(x, 0)).sum(axis=1)
        df['PHQ-12'] = df.apply(lambda row: None if any(pd.isna(row[col]) for col in phq12_columns[:-2]) else row['PHQ-12'], axis=1)
    else:
        print("Some columns are missing in the dataframe.")
    return df

# Apply the function to each dataframe
HC_reference_clinical = calculate_phq12_score(HC_reference_clinical)
HC_clinical = calculate_phq12_score(HC_clinical)
IBS_clinical = calculate_phq12_score(IBS_clinical)

### GAD-7

In [5]:
# Define the columns based on the UK Biobank coding for GAD-7
gad7_columns = [
    '20506-0.0',  # Nervous
    '20509-0.0',  # Uncontrollable worry
    '20520-0.0',  # Excess worry
    '20515-0.0',  # Trouble relaxing
    '20516-0.0',  # Restless
    '20505-0.0',  # Irritable
    '20512-0.0'   # Foreboding
]

# Mapping of responses to scores
response_to_score_gad7 = {
    1: 0,  # Not at all
    2: 1,  # Several days
    3: 2,  # More than half of the days
    4: 3   # Nearly every day
}

# Function to calculate GAD-7 score
def calculate_gad7_score(df):
    # Check if all required columns are present
    if all(col in df.columns for col in gad7_columns):
        # Map responses to scores and sum them
        df['GAD-7'] = df[gad7_columns].applymap(lambda x: response_to_score_gad7.get(x, 0)).sum(axis=1)
        df['GAD-7'] = df.apply(lambda row: None if any(pd.isna(row[col]) for col in gad7_columns) else row['GAD-7'], axis=1)
    else:
        print("Some columns are missing in the dataframe.")
    return df

# Apply the function to each dataframe
HC_reference_clinical = calculate_gad7_score(HC_reference_clinical)
HC_clinical = calculate_gad7_score(HC_clinical)
IBS_clinical = calculate_gad7_score(IBS_clinical)

### PHQ-9

In [6]:
# Define the columns based on the UK Biobank coding for PHQ-9
phq9_columns = [
    '20514-0.0',  # Anhedonia
    '20510-0.0',  # Feeling down
    '20517-0.0',  # Sleep disorder
    '20519-0.0',  # Tired
    '20511-0.0',  # Appetite disorder
    '20507-0.0',  # Feeling inadequate
    '20508-0.0',  # Concentration
    '20518-0.0',  # Tach- or bradykinetic
    '20513-0.0'   # Self-harm
]

# Mapping of responses to scores
response_to_score_phq9 = {
    1: 0,  # Not at all
    2: 1,  # Several days
    3: 2,  # More than half of the days
    4: 3   # Nearly every day
}

# Function to calculate PHQ-9 score
def calculate_phq9_score(df):
    # Check if all required columns are present
    if all(col in df.columns for col in phq9_columns):
        # Map responses to scores and sum them
        df['PHQ-9'] = df[phq9_columns].applymap(lambda x: response_to_score_phq9.get(x, 0)).sum(axis=1)
        df['PHQ-9'] = df.apply(lambda row: None if any(pd.isna(row[col]) for col in phq9_columns) else row['PHQ-9'], axis=1)
    else:
        print("Some columns are missing in the dataframe.")
    return df

# Apply the function to each dataframe
HC_reference_clinical = calculate_phq9_score(HC_reference_clinical)
HC_clinical = calculate_phq9_score(HC_clinical)
IBS_clinical = calculate_phq9_score(IBS_clinical)

In [7]:
HC_reference_clinical.to_csv(os.path.join(data_dir, 'HC_reference_clinical.csv'),index=False)
HC_clinical.to_csv(os.path.join(data_dir, 'HC_clinical.csv'),index=False)
IBS_clinical.to_csv(os.path.join(data_dir, 'IBS_clinical.csv'),index=False)

# Demographic Table

In [8]:
# Function to calculate demographic statistics
def calculate_demographics(data, group_name):

    # Age
    age_data = data['age'].dropna()
    mean_age = age_data.mean()
    std_age = age_data.std()
    n_age = len(age_data)
    
    # BMI
    bmi_data = data['BMI'].dropna()
    mean_bmi = bmi_data.mean()
    std_bmi = bmi_data.std()
    n_bmi = len(bmi_data)
    # if possible
    clinical_data = data[['IBS-SSS', 'PHQ-12', 'PHQ-9', 'GAD-7']].dropna()
    # IBS-SSS
    ibsss_data = clinical_data['IBS-SSS']
    mean_ibsss = ibsss_data.mean()
    std_ibsss = ibsss_data.std()
    n_ibsss = len(ibsss_data)
    
    # PHQ-12
    phq12_data = clinical_data['PHQ-12']
    mean_phq12 = phq12_data.mean()
    std_phq12 = phq12_data.std()
    n_phq12 = len(phq12_data)
    
    # GAD-7
    gad7_data = clinical_data['GAD-7']
    mean_gad7 = gad7_data.mean()
    std_gad7 = gad7_data.std()
    n_gad7 = len(gad7_data)
    
    # PHQ-9
    phq9_data = clinical_data['PHQ-9']
    mean_phq9 = phq9_data.mean()
    std_phq9 = phq9_data.std()
    n_phq9 = len(phq9_data)
    female_count = (data['sex'] == 0).sum()
    male_count = (data['sex'] == 1).sum()
    female_percentage = (female_count / len(data)) * 100
    male_percentage = (male_count / len(data)) * 100
    num_data_points = len(data)
    return {
        'Group': group_name,
        'Mean Age': mean_age,
        'Std Age': std_age,
        'N Age': n_age,
        'Mean BMI': mean_bmi,
        'Std BMI': std_bmi,
        'N BMI': n_bmi,
        'Mean IBS-SSS': mean_ibsss,
        'Std IBS-SSS': std_ibsss,
        'N IBS-SSS': n_ibsss,
        'Mean PHQ-12': mean_phq12,
        'Std PHQ-12': std_phq12,
        'N PHQ-12': n_phq12,
        'Mean GAD-7': mean_gad7,
        'Std GAD-7': std_gad7,
        'N GAD-7': n_gad7,
        'Mean PHQ-9': mean_phq9,
        'Std PHQ-9': std_phq9,
        'N PHQ-9': n_phq9,
        'Female Count': female_count,
        'Male Count': male_count,
        'Female %': female_percentage,
        'Male %': male_percentage,
        'Number of Data Points': num_data_points
    }

In [9]:
# Load clinical data
HC_reference_clinical = pd.read_csv(os.path.join(data_dir, 'HC_reference_clinical.csv'))
HC_clinical = pd.read_csv(os.path.join(data_dir, 'HC_clinical.csv'))
IBS_clinical = pd.read_csv(os.path.join(data_dir, 'IBS_clinical.csv'))

# Create subgroup data
IBS_subgroups = {
    'IBS ROME III': IBS_clinical[IBS_clinical['Group'] == 1].copy(),
    'IBS Diagnosed': IBS_clinical[IBS_clinical['Group'] == 2].copy(),
    'IBS Both': IBS_clinical[IBS_clinical['Group'] == 3].copy()
}

# Define all groups to process
clinical_groups = {
    'HC Reference': HC_reference_clinical,
    'HC Case': HC_clinical,
    'HC Female': HC_clinical[HC_clinical['sex'] == 0],
    'HC Male': HC_clinical[HC_clinical['sex'] == 1],
    'IBS All': IBS_clinical,
    'IBS Female': IBS_clinical[IBS_clinical['sex'] == 0],
    'IBS Male': IBS_clinical[IBS_clinical['sex'] == 1]
}

# Add subgroup stats
for label, df in IBS_subgroups.items():
    clinical_groups[label] = df
    clinical_groups[f"{label} Female"] = df[df['sex'] == 0]
    clinical_groups[f"{label} Male"] = df[df['sex'] == 1]

# Calculate all demographic summaries
clinical_stats = {label: calculate_demographics(df, label) for label, df in clinical_groups.items()}

# Build summary table
columns = ['N', 'Female %', 'Age (mean (SD))', 'BMI (mean (SD))', 'N IBS-SSS',
           'IBS-SSS (mean (SD))', 'PHQ-12 (mean (SD))', 'GAD-7 (mean (SD))', 'PHQ-9 (mean (SD))']

summary_data = []
for label in clinical_stats:
    stats = clinical_stats[label]
    summary_data.append([
        stats['N Age'], f"{stats['Female %']:.1f}",
        f"{stats['Mean Age']:.1f} ({stats['Std Age']:.1f})",
        f"{stats['Mean BMI']:.1f} ({stats['Std BMI']:.1f})",
        stats['N IBS-SSS'],
        f"{stats['Mean IBS-SSS']:.1f} ({stats['Std IBS-SSS']:.1f})",
        f"{stats['Mean PHQ-12']:.1f} ({stats['Std PHQ-12']:.1f})",
        f"{stats['Mean GAD-7']:.1f} ({stats['Std GAD-7']:.1f})",
        f"{stats['Mean PHQ-9']:.1f} ({stats['Std PHQ-9']:.1f})"
    ])

# Create DataFrame
clinical_summary_df = pd.DataFrame(summary_data, columns=columns, index=clinical_stats.keys())

# Save
clinical_summary_df.to_csv(os.path.join(data_dir, 'clinical_summary.csv'))

## Statistical comparison

In [10]:
from numpy import mean, var, sqrt
from scipy.stats import ttest_ind, levene, chi2_contingency

def cohen_d(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    s1, s2 = var(group1, ddof=1), var(group2, ddof=1)
    pooled_std = sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
    return (mean(group1) - mean(group2)) / pooled_std

def ttest_with_effect(group1, group2):
    """Run Levene’s test then t-test and Cohen's d."""
    group1, group2 = group1.dropna(), group2.dropna()
    if len(group1) < 2 or len(group2) < 2:
        return float('nan'), float('nan')  # Not enough data
    _, p_levene = levene(group1, group2)
    t_stat, p_value = ttest_ind(group1, group2, equal_var=(p_levene > 0.05))
    return p_value, cohen_d(group1, group2)

def perform_stat_tests(group_data, hc_data, group_name):
    # Continuous variables
    age_p, age_d = ttest_with_effect(group_data['age'], hc_data['age'])
    bmi_p, bmi_d = ttest_with_effect(group_data['BMI'], hc_data['BMI'])

    # Categorical: Sex (chi-squared)
    contingency = pd.crosstab(group_data['sex'], hc_data['sex'])
    _, sex_p, _, _ = chi2_contingency(contingency)

    # Questionnaire scores
    ibsss_p, ibsss_d = ttest_with_effect(group_data['IBS-SSS'], hc_data['IBS-SSS'])
    phq12_p, phq12_d = ttest_with_effect(group_data['PHQ-12'], hc_data['PHQ-12'])
    gad7_p, gad7_d = ttest_with_effect(group_data['GAD-7'], hc_data['GAD-7'])
    phq9_p, phq9_d = ttest_with_effect(group_data['PHQ-9'], hc_data['PHQ-9'])

    return {
        'Group': group_name,
        'Age p-value': age_p, 'Age Effect Size': age_d,
        'BMI p-value': bmi_p, 'BMI Effect Size': bmi_d,
        'Sex p-value': sex_p,
        'IBS-SSS p-value': ibsss_p, 'IBS-SSS Effect Size': ibsss_d,
        'PHQ-12 p-value': phq12_p, 'PHQ-12 Effect Size': phq12_d,
        'GAD-7 p-value': gad7_p, 'GAD-7 Effect Size': gad7_d,
        'PHQ-9 p-value': phq9_p, 'PHQ-9 Effect Size': phq9_d
    }

# Run statistical comparisons
stat_tests = [
    perform_stat_tests(IBS_clinical, HC_clinical, 'IBS All'),
    perform_stat_tests(IBS_clinical[IBS_clinical['Group'] == 1], HC_clinical, 'IBS ROME III'),
    perform_stat_tests(IBS_clinical[IBS_clinical['Group'] == 2], HC_clinical, 'IBS Diagnosed'),
    perform_stat_tests(IBS_clinical[IBS_clinical['Group'] == 3], HC_clinical, 'IBS Both'),
]

# Convert to DataFrame
stat_tests_df = pd.DataFrame(stat_tests)
stat_tests_df.to_csv(os.path.join(data_dir, 'clinical_summary_stats.csv'))

# Correlation analyses

## Correlation comparison between HC and IBS subgroups

### Correlation calculation

In [11]:
from utils_norm.utils_analyses import calculate_correlation_and_z_scores, calculate_and_compare_correlations

In [12]:
demo_columns = ['IBS-SSS', 'PHQ-12', 'PHQ-9', 'GAD-7']
HC_clinical = pd.read_csv(os.path.join(data_dir, 'HC_clinical.csv'))
IBS_clinical = pd.read_csv(os.path.join(data_dir, 'IBS_clinical.csv'))
ROI_list = pd.read_csv(os.path.join(docu_dir, 'ROI_IBS.csv'))
HC_clinical.rename(columns={'31-0.0':'sex'}, inplace=True)
IBS_clinical.rename(columns={'31-0.0':'sex'}, inplace=True)

# Loop through deviation types
for file_name in ['CT', 'SA', 'CV']:
    deviation_dir = os.path.join(model_dir, f"{file_name}_age_45_85", f"perm_{perm}")
    HC_deviation = pd.read_csv(os.path.join(deviation_dir, 'HC_deviation.csv'))
    IBS_deviation = pd.read_csv(os.path.join(deviation_dir, 'Patient_deviation_IBS.csv'))
    IBS_deviation = IBS_deviation.merge(IBS_clinical[['eid', 'sex', 'Group']], on='eid', how='left')

    HC_all = pd.merge(HC_clinical[['eid']+demo_columns], HC_deviation, on='eid', how='inner')
    IBS_all = pd.merge(IBS_clinical[['eid']+demo_columns], IBS_deviation, on='eid', how='inner')

    # Split IBS deviation into subgroups
    IBS_deviation_subgroup = {
        'IBS ROME III': IBS_all[IBS_all['Group'] == 1].copy(),
        'IBS Diagnosed': IBS_all[IBS_all['Group'] == 2].copy(),
        'IBS Both': IBS_all[IBS_all['Group'] == 3].copy()
    }

    # Compute correlation data for each group
    corr_data = {
        'HC': calculate_correlation_and_z_scores(HC_all, ROI_list, demo_columns),
        
        'IBS ROME III': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS ROME III'], ROI_list, demo_columns),
        'IBS ROME III Female': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS ROME III'][IBS_deviation_subgroup['IBS ROME III']['sex'] == 0], ROI_list, demo_columns),
        'IBS ROME III Male': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS ROME III'][IBS_deviation_subgroup['IBS ROME III']['sex'] == 1], ROI_list, demo_columns),

        'IBS Diagnosed': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Diagnosed'], ROI_list, demo_columns),
        'IBS Diagnosed Female': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Diagnosed'][IBS_deviation_subgroup['IBS Diagnosed']['sex'] == 0], ROI_list, demo_columns),
        'IBS Diagnosed Male': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Diagnosed'][IBS_deviation_subgroup['IBS Diagnosed']['sex'] == 1], ROI_list, demo_columns),

        'IBS Both': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Both'], ROI_list, demo_columns),
        'IBS Both Female': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Both'][IBS_deviation_subgroup['IBS Both']['sex'] == 0], ROI_list, demo_columns),
        'IBS Both Male': calculate_correlation_and_z_scores(
            IBS_deviation_subgroup['IBS Both'][IBS_deviation_subgroup['IBS Both']['sex'] == 1], ROI_list, demo_columns),
    }

    # Compare each IBS group to HC across all clinical variables
    for demo_str in demo_columns:
        results_dir = os.path.join(out_dir, demo_str)
        os.makedirs(results_dir, exist_ok=True)

        for group_label in ['IBS ROME III', 'IBS Diagnosed', 'IBS Both']:
            results = calculate_and_compare_correlations(corr_data['HC'], corr_data[group_label], demo_str, 'HC', 'IBS')
            results.to_csv(os.path.join(results_dir, f'correlation_results_{group_label.replace(" ", "_")}_{file_name}.csv'))

            results = calculate_and_compare_correlations(corr_data[group_label+' Female'], corr_data[group_label+' Male'], demo_str, 'IBS Female', 'IBS Male')
            results.to_csv(os.path.join(results_dir, f'correlation_results_{group_label.replace(" ", "_")}_{file_name}_sex.csv'))

### Visualization

In [13]:
import numpy as np
import matplotlib.pyplot as plt

In [14]:
def plot_significant_correlations(sig_results, group_1_label, group_2_label, hc_data, ibs_data, demo_str, results_dir):
    
    for _, row in sig_results.iterrows():
        roi = row['ROI']
        hc_data.dropna(inplace=True)
        ibs_data.dropna(inplace=True)
        # Create scatter plot
        plt.figure(figsize=(8, 6))
        plt.grid(True)
        # Plot HC data
        hc_demo_rank = (hc_data[demo_str].rank() - hc_data[demo_str].rank().min()) / (hc_data[demo_str].rank().max() - hc_data[demo_str].rank().min()) * (hc_data[demo_str].max() - hc_data[demo_str].min()) + hc_data[demo_str].min()
        hc_roi_rank = (hc_data[roi].rank() - hc_data[roi].rank().min()) / (hc_data[roi].rank().max() - hc_data[roi].rank().min()) * (hc_data[roi].max() - hc_data[roi].min()) + hc_data[roi].min()
        plt.scatter(hc_data[demo_str], hc_data[roi], 
                   label=f'{group_1_label} (r = {row[group_1_label+"_r"]:.3f}, p_fdr = {row[group_1_label+"_p_fdr"]:.3f})',
                   alpha=0.2, color='blue')
        z = np.polyfit(hc_demo_rank, hc_roi_rank, 1)
        p = np.poly1d(z)
        plt.plot(hc_demo_rank, p(hc_demo_rank), color='blue', alpha=0.9)
        
        # Plot IBS data
        ibs_demo_rank = (ibs_data[demo_str].rank() - ibs_data[demo_str].rank().min()) / (ibs_data[demo_str].rank().max() - ibs_data[demo_str].rank().min()) * (ibs_data[demo_str].max() - ibs_data[demo_str].min()) + ibs_data[demo_str].min()
        ibs_roi_rank = (ibs_data[roi].rank() - ibs_data[roi].rank().min()) / (ibs_data[roi].rank().max() - ibs_data[roi].rank().min()) * (ibs_data[roi].max() - ibs_data[roi].min()) + ibs_data[roi].min()
        plt.scatter(ibs_data[demo_str], ibs_data[roi], 
                   label=f'{group_2_label} (r = {row[group_2_label+"_r"]:.3f}, p_fdr = {row[group_2_label+"_p_fdr"]:.3f})',
                   alpha=0.2, color='red')
        z = np.polyfit(ibs_demo_rank, ibs_roi_rank, 1)
        p = np.poly1d(z)
        plt.plot(ibs_demo_rank, p(ibs_demo_rank), color='red', alpha=0.9)
        
        # Add diff_p_fdr annotation
        plt.annotate(f'diff_p_fdr = {row["diff_p_fdr"]:.3f}', 
                    xy=(0.98, 0.98), 
                    xycoords='axes fraction',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'),
                    verticalalignment='top',
                    horizontalalignment='right')
        
        plt.xlabel(f'{demo_str}')
        plt.ylabel(f'{roi}')
        # plt.title(f'Correlation between {roi} and {demo_str}\nGroup {group_num}', pad=20)
        plt.legend(bbox_to_anchor=(0.5, 1.15), loc='upper center', ncol=2)
        
        # Save plot with bbox_inches='tight' to include the legend
        plt.savefig(f'{results_dir}_correlation_plot_{group_1_label.replace(" ", "_")}_{group_2_label.replace(" ", "_")}_{roi}_{demo_str}.svg', 
            format='svg', dpi=300, bbox_inches='tight')
        plt.savefig(f'{results_dir}_correlation_plot_{group_1_label.replace(" ", "_")}_{group_2_label.replace(" ", "_")}_{roi}_{demo_str}.png', 
                   bbox_inches='tight')
        plt.close()

In [ ]:
demo_columns = ['IBS-SSS', 'PHQ-12', 'PHQ-9', 'GAD-7']
HC_clinical = pd.read_csv(os.path.join(data_dir, 'HC_clinical.csv'))
IBS_clinical = pd.read_csv(os.path.join(data_dir, 'IBS_clinical.csv'))
ROI_list = pd.read_csv(os.path.join(docu_dir, 'ROI_IBS.csv'))
HC_clinical.rename(columns={'31-0.0':'sex'}, inplace=True)
IBS_clinical.rename(columns={'31-0.0':'sex'}, inplace=True)
IBS_subgroup_label = ['IBS ROME III', 'IBS Diagnosed', 'IBS Both']
# Loop through deviation types
for file_name in ['CT', 'SA', 'CV']:
    deviation_dir = os.path.join(model_dir, f"{file_name}_age_45_85", f"perm_{perm}")
    HC_deviation = pd.read_csv(os.path.join(deviation_dir, 'HC_deviation.csv'))
    HC_deviation.dropna(how='all', axis=1, inplace=True)
    IBS_deviation = pd.read_csv(os.path.join(deviation_dir, 'Patient_deviation_IBS.csv'))
    IBS_deviation = IBS_deviation.merge(IBS_clinical[['eid', 'sex', 'Group']], on='eid', how='left')
    IBS_deviation.dropna(how='all', axis=1, inplace=True)

    HC_all = pd.merge(HC_clinical[['eid']+demo_columns], HC_deviation, on='eid', how='inner')
    IBS_all = pd.merge(IBS_clinical[['eid']+demo_columns], IBS_deviation, on='eid', how='inner')

    # Split IBS deviation into subgroups
    IBS_deviation_subgroup = {
        'IBS ROME III': IBS_all[IBS_all['Group'] == 1].copy(),
        'IBS Diagnosed': IBS_all[IBS_all['Group'] == 2].copy(),
        'IBS Both': IBS_all[IBS_all['Group'] == 3].copy()
    }

    for demo_str in demo_columns:
        for IBS_subgroup in IBS_subgroup_label:
            results_dir = os.path.join(out_dir, demo_str)
            results = pd.read_csv(os.path.join(results_dir, f'correlation_results_{IBS_subgroup.replace(" ", "_")}_{file_name}.csv'))
            results = results.rename(columns={results.columns[0]: 'ROI'})
            # Filter for significant results
            sig_results = results[(results['diff_p_fdr'] < 0.05) & (results['IBS_p_fdr'] < 0.05)]
            if not sig_results.empty:
                plot_significant_correlations(sig_results, 'HC', 'IBS', HC_all, IBS_deviation_subgroup[IBS_subgroup], demo_str, os.path.join(results_dir, f'{file_name}_{IBS_subgroup.replace(" ", "_")}'))
            results = pd.read_csv(os.path.join(results_dir, f'correlation_results_{IBS_subgroup.replace(" ", "_")}_{file_name}_sex.csv'))
            results = results.rename(columns={results.columns[0]: 'ROI'})
            # Filter for significant results
            sig_results = results[(results['diff_p_fdr'] < 0.05) & ((results['IBS Female_p_fdr'] < 0.05) | (results['IBS Male_p_fdr'] < 0.05))]
            if not sig_results.empty:
                plot_significant_correlations(sig_results, f'IBS Female', f'IBS Male', IBS_deviation_subgroup[IBS_subgroup][IBS_deviation_subgroup[IBS_subgroup]['sex'] == 0].copy(), 
                                              IBS_deviation_subgroup[IBS_subgroup][IBS_deviation_subgroup[IBS_subgroup]['sex'] == 1].copy(), demo_str, os.path.join(results_dir, f'sex_diff_{file_name}_{IBS_subgroup.replace(" ", "_")}'))
            